In [ ]:


import hydra
import torch
import numpy as np
from pathlib import Path
from plyfile import PlyData, PlyElement
from torch.utils.data import DataLoader
import MinkowskiEngine as ME
#from trainer.trainer import InstanceSegmentation
import albumentations as A
from utils.utils import (
    load_checkpoint_with_missing_or_exsessive_keys,
    load_backbone_checkpoint_with_missing_or_exsessive_keys,
)
from utils.utils import (
    load_checkpoint_with_missing_or_exsessive_keys,
    load_backbone_checkpoint_with_missing_or_exsessive_keys,
)
 



/home/cnhemwa/miniforge3/envs/human3d_fin/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/cnhemwa/miniforge3/envs/human3d_fin/lib/python3.10/site-packages/MinkowskiEngine/__init__.py:36: UserWarning: The environment variable `OMP_NUM_THREADS` not set. MinkowskiEngine will automatically set `OMP_NUM_THREADS=16`. If you want to set `OMP_NUM_THREADS` manually, please export it on the command line before running a python script. e.g. `export OMP_NUM_THREADS=12; python your_program.py`. It is recommended to set it below 24.
  warnings.warn(
/home/cnhemwa/miniforge3/envs/human3d_fin/lib/python3.10/site-packages/pytorch_lightning/utilities/imports.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for

In [ ]:
class InstanceSegmentation(torch.nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.model = hydra.utils.instantiate(cfg.model)


    def forward(
        self,
        x,
        point2segment=None,
        raw_coordinates=None,
        is_eval=True,
        clip_feat=None,
        clip_pos=None,
    ):
        x = self.model(
            x,
            point2segment,
            raw_coordinates=raw_coordinates,
            is_eval=is_eval,
            clip_feat=clip_feat,
            clip_pos=clip_pos,
        )
        return x
    

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [3]:
def get_model(checkpoint_path=None):
    from hydra.experimental import initialize, compose

    with initialize(config_path="conf"):
        cfg = compose(config_name="config_base_instance_segmentation.yaml")

    cfg.general.checkpoint = checkpoint_path
    cfg.general.experiment_name = "Mask3D_horse_eval"
    cfg.general.project_name = "mask3d_horse_seg"
    cfg.general.num_targets = 16
    cfg.data.num_labels = 16
    cfg.model.num_human_queries = 5
    cfg.model.num_parts_per_human_queries = 16
    cfg.trainer.check_val_every_n_epoch = 1
    cfg.general.topk_per_image = -1
    cfg.model.non_parametric_queries = False
    cfg.trainer.max_epochs = 36
    cfg.data.batch_size = 4
    cfg.data.num_workers = 10
    cfg.general.reps_per_epoch = 1
    cfg.model.config.backbone._target_ = "models.Res16UNet18B"
    cfg.general.train_mode = False
    cfg.general.save_visualizations = True

    model = InstanceSegmentation(cfg)

    if cfg.general.backbone_checkpoint is not None:
        cfg, model = load_backbone_checkpoint_with_missing_or_exsessive_keys(cfg, model)
    if cfg.general.checkpoint is not None:
        cfg, model = load_checkpoint_with_missing_or_exsessive_keys(cfg, model)

    return model

In [7]:
model = get_model('/oslab-data-4tb/Github/Human3D/checkpoints/horse_mask.ckpt')
# Switch model to evaluation mode
model.eval()

model.to(device)

2025-07-29 15:14:42.644 | WARNING  | utils.utils:load_checkpoint_with_missing_or_exsessive_keys:91 - Key not found, it will be initialized randomly: model.class_embed_head.weight
2025-07-29 15:14:42.645 | WARNING  | utils.utils:load_checkpoint_with_missing_or_exsessive_keys:91 - Key not found, it will be initialized randomly: model.class_embed_head.bias
2025-07-29 15:14:42.855 | WARNING  | utils.utils:load_checkpoint_with_missing_or_exsessive_keys:103 - incorrect shape model.backbone.final.kernel:torch.Size([128, 2]) vs torch.Size([128, 16])
2025-07-29 15:14:42.857 | WARNING  | utils.utils:load_checkpoint_with_missing_or_exsessive_keys:103 - incorrect shape model.backbone.final.bias:torch.Size([1, 2]) vs torch.Size([1, 16])
2025-07-29 15:14:42.858 | WARNING  | utils.utils:load_checkpoint_with_missing_or_exsessive_keys:103 - incorrect shape model.query_feat.weight:torch.Size([1, 128]) vs torch.Size([85, 128])
2025-07-29 15:14:42.859 | WARNING  | utils.utils:load_checkpoint_with_missing_

InstanceSegmentation(
  (model): Mask3DHumanParts(
    (backbone): Res16UNet18B(
      (conv0p1s1): MinkowskiConvolution(in=3, out=32, kernel_size=[5, 5, 5], stride=[1, 1, 1], dilation=[1, 1, 1])
      (bn0): MinkowskiBatchNorm(32, eps=1e-05, momentum=0.02, affine=True, track_running_stats=True)
      (conv1p1s2): MinkowskiConvolution(in=32, out=32, kernel_size=[2, 2, 2], stride=[2, 2, 2], dilation=[1, 1, 1])
      (bn1): MinkowskiBatchNorm(32, eps=1e-05, momentum=0.02, affine=True, track_running_stats=True)
      (block1): Sequential(
        (0): BasicBlock(
          (conv1): MinkowskiConvolution(in=32, out=32, kernel_size=[3, 3, 3], stride=[1, 1, 1], dilation=[1, 1, 1])
          (norm1): MinkowskiBatchNorm(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (conv2): MinkowskiConvolution(in=32, out=32, kernel_size=[3, 3, 3], stride=[1, 1, 1], dilation=[1, 1, 1])
          (norm2): MinkowskiBatchNorm(32, eps=1e-05, momentum=0.1, affine=True, track_running_s

In [8]:
def read_ply_file(ply_file_path):
    ply_data = PlyData.read(ply_file_path)
    vertex_data = ply_data['vertex']
   
    # Extract the 3D coordinates and other relevant information (like colors and normals)
    full_res_coords = np.array([list(vertex) for vertex in zip(vertex_data['x'], vertex_data['y'], vertex_data['z'])])
    original_colors = np.array([list(vertex) for vertex in zip(vertex_data['red'], vertex_data['green'], vertex_data['blue'])])
    # Optionally extract normals if available
    if 'nx' in vertex_data and 'ny' in vertex_data and 'nz' in vertex_data:
        original_normals = np.array([list(vertex) for vertex in zip(vertex_data['nx'], vertex_data['ny'], vertex_data['nz'])])
    else:
        original_normals = None  # If normals are not available in the .ply file
   
    return full_res_coords, original_colors, original_normals

In [ ]:
# Specify the path of the .ply file
ply_file_path = "/oslab-data-4tb/Github/Human3D/test3.ply" 
  # Read
full_res_coords, original_colors, original_normals = read_ply_file(ply_file_path) and prepare the data from the .ply file

In [45]:
full_res_coords

array([[-0.32915902, -0.12831021, -0.70100202],
       [ 1.58511772,  0.41616689, -0.16904299],
       [ 0.37463297,  0.20585374, -0.04678433],
       ...,
       [ 1.91144343,  0.58421385, -0.83644677],
       [ 1.97492963,  0.61046565, -0.84854983],
       [ 1.8734879 ,  0.59994766, -0.87222378]])

In [48]:

def prepare_data(points, colors, device):
    color_mean = (0.47793125906962, 0.4303257521323044, 0.3749598901421883)
    color_std = (0.2834475483823543, 0.27566157565723015, 0.27018971370874995)
    normalize_color = A.Normalize(mean=color_mean, std=color_std)

    pseudo_image = colors[np.newaxis, :, :]
   
    coords = np.floor(points / 0.02)
    _, _, unique_map, inverse_map = ME.utils.sparse_quantize(
        coordinates=torch.from_numpy(coords).contiguous(),
        features=colors,
        return_index=True,
        return_inverse=True,
    )

    sample_coordinates = coords[unique_map]
    coordinates = [torch.from_numpy(sample_coordinates).int()]
    sample_features = colors[unique_map]
    features = [torch.from_numpy(sample_features).float()]

    coordinates, _ = ME.utils.sparse_collate(coords=coordinates, feats=features)
    features = torch.cat(features, dim=0)

    sample_features = sample_coordinates  # Use XYZ as features
    features = [torch.from_numpy(sample_features).float()]
    features = [(f - f.mean(dim=0)) / (f.std(dim=0) + 1e-5) for f in features]

    features = torch.cat(features, dim=0)


    data = ME.SparseTensor(coordinates=coordinates, features=features, device=device)

    return data, points, colors, features, unique_map, inverse_map

In [47]:

def prepare_data_for_model(full_res_coords, original_colors, original_normals):
    # Example of converting data into the required format for your model.
    # If your model uses SparseTensor, you might need to use the MinkowskiEngine here.
    # For the sake of simplicity, we'll just return a tensor with the coordinates for now.

    # This might need adjustment based on your model's data format.
    coordinates = torch.tensor(full_res_coords, dtype=torch.float32)  # Convert to tensor
    features = torch.tensor(original_colors, dtype=torch.float32)  # You may need to normalize or process features

    # Create a SparseTensor or whatever your model requires
    data = ME.SparseTensor(coordinates=coordinates, features=features, device=torch.device("cuda" if torch.cuda.is_available() else "cpu"))
   
    return data, features


In [69]:
def map_output_to_pointcloud(outputs, inverse_map, num_vertices, label_space='scannet200', confidence_threshold=0.9):
    logits = outputs["pred_human_logits"][0].detach().cpu()
    masks = outputs["pred_masks"][0].detach().cpu()

    labels_mapped = np.zeros((num_vertices, 1))
    for i in range(len(logits)):
        p_labels = torch.softmax(logits[i], dim=-1)
        p_masks = torch.sigmoid(masks[:, i])
        l = torch.argmax(p_labels, dim=-1)
        c_label = torch.max(p_labels)
        m = p_masks > 0.01
        c_m = p_masks[m].sum() / (m.sum() + 1e-8)
        c = c_label * c_m
        if l < 200 and c > confidence_threshold:
            label_offset = 1 if label_space == 'scannet200' else 0
            labels_mapped[m[inverse_map].numpy()] = int(l) + label_offset

    return labels_mapped

In [49]:
data,features = prepare_data_for_model(full_res_coords, original_colors, original_normals)

/home/cnhemwa/miniforge3/envs/human3d_fin/lib/python3.10/site-packages/MinkowskiEngine/MinkowskiSparseTensor.py:295: UserWarning: coordinates implicitly converted to torch.IntTensor. To remove this warning, use `.int()` to convert the coords into an torch.IntTensor
  warnings.warn(


In [76]:
coords = np.vstack([np.array(full_res_coords)])

# Label column 0 is instance, column 1 is part of horse.
horse_labels = np.ones((np.array(full_res_coords).shape[0],2 ), dtype=np.int32)
other_labels = np.zeros((coords.shape[0] - horse_labels.shape[0], 2), dtype=np.int32)
labels = np.concatenate((horse_labels, other_labels))
rgb = np.zeros_like(coords)
process_data = np.hstack((coords, rgb, labels))

In [50]:
data , points, colors_norm, features, unique_map, inverse_map  = prepare_data(full_res_coords, original_colors, device)

In [ ]:
with torch.no_grad():
        output = model.forward(data, raw_coordinates=features, is_eval=True)


AssertionError: 

In [72]:
labels = map_output_to_pointcloud(output, inverse_map, len(points))

In [74]:
labels

array([[0.],
       [0.],
       [0.],
       ...,
       [0.],
       [0.],
       [0.]])

In [53]:

pred_mask = output['pred_masks']
pred_human_logits  = output["pred_human_logits"]

In [56]:
pred_human_logits.shape

torch.Size([1, 5, 2])

In [67]:
unique_labels, counts = np.unique(labels, return_counts=True)

In [68]:
unique_labels

array([0.])

In [4]:
import numpy as np

np.load("./data/horse/processed/validation/H00018_still1.000025_labels.npy").shape

(118463, 8)